# Python Bridge for 09b

## A Short Code Refresher

Use this optional notebook alongside 09b to practice fitting an anomaly detector, reading its scores, and keeping those scores attached to the correct observations. Allow about 15–20 minutes. The seven values from the lecture are supplied here, so no data download is needed.

The lecture develops why LOF and Isolation Forest can disagree. Here we inspect the Python objects behind that comparison: a two-dimensional input table, fitted attributes, returned arrays, ranks, and selected rows.

## References

The scikit-learn references for [LocalOutlierFactor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html) and [IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html) describe the score signs and interfaces used below. The [anomaly-detection guide](https://scikit-learn.org/stable/modules/outlier_detection.html) explains the separate novelty-detection workflow for new observations.

## Keep One Column Two-Dimensional

Each row is an invented observation of one numerical feature. The row letters identify records; the values determine distances. Scikit-learn expects a table of rows and features even when there is only one feature.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

records = pd.DataFrame({"value": [0, 1, 3, 10, 50, 60, 80]}, index=list("ABCDEFG"))
X = records[["value"]]
print("Series shape:", records["value"].shape)
print("Model input shape:", X.shape)
display(X)

One pair of brackets selects a Series with shape `(7,)`. The list inside the second pair selects a DataFrame with shape `(7, 1)`. Keep the row index so a returned score can be traced back to its input.

## Read LOF's Fitted Scores

Create a model using two neighbors, matching the lecture example, then fit it to these seven observations. `negative_outlier_factor_` is an attribute learned during fitting. Negating it gives the lecture's orientation: larger means more unusual relative to nearby observations.

In [ ]:
lof = LocalOutlierFactor(n_neighbors=2, metric="euclidean")
lof.fit(X)
lof_scores = pd.Series(-lof.negative_outlier_factor_, index=X.index, name="LOF")

display(records.join(lof_scores).round(3))

D, with value 10, scores about 2.933. It is less densely supported than its nearby reference observations. G, with value 80, scores about 0.917 despite having a larger coordinate. An anomaly score is not the original feature value or a probability of a fault.

## Call Isolation Forest's Scoring Method

Isolation Forest provides `score_samples(X)` as a method. Its native orientation also gives lower scores to more unusual observations, so negate the result before comparing ranks. The fixed seed makes this example repeatable; 200 trees average over many random partitions.

In [ ]:
forest = IsolationForest(n_estimators=200, random_state=7130)
forest.fit(X)
forest_scores = pd.Series(-forest.score_samples(X), index=X.index, name="Isolation Forest")
scores = pd.concat([lof_scores, forest_scores], axis=1)

display(records.join(scores).round(3))

Each result array follows the supplied row order. The Series constructors restore the row letters before `concat` aligns the columns by index. Both columns now increase with unusualness, but their numerical scales differ.

## Compare Ranks and Select Rows

Rank each column separately. `ascending=False` gives rank 1 to the largest score. Tied values receive an average rank by default. For a concrete selection, obtain the two highest LOF scores with `nlargest`, then use their row labels with `.loc`.

In [ ]:
ranks = scores.rank(ascending=False)
display(ranks)
top_rows = lof_scores.nlargest(2).index
display(records.loc[top_rows].join(scores.loc[top_rows]).round(3))

LOF places D first; Isolation Forest places G first. Their disagreement follows the different comparisons taught in 09b. The two-row selection is an illustrative review list, not a validated alert rule. If a score ties at the boundary, selecting an exact count needs a tie rule; a score threshold can include every tied observation.

Plot the two score columns separately so their different scales remain visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout="constrained")
for ax, name in zip(axes, scores.columns, strict=True):
    ax.bar(scores.index, scores[name], color="steelblue")
    ax.set(title=name, xlabel="Observation", ylabel="Oriented anomaly score")
plt.show()

## Inspect Which Interface You Are Using

The default LOF workflow scores the observations used in fitting through `negative_outlier_factor_`. The object does not expose `predict` in this mode. Isolation Forest can score or predict supplied rows after fitting. Its `predict` method returns `1` or `-1` using the model's own cutoff, rather than our two-row review rule.

In [ ]:
print("Default LOF exposes predict:", hasattr(lof, "predict"))
display(records.assign(forest_label=forest.predict(X)))

For a later task involving new observations, LOF needs a separately fitted model with `novelty=True` and its documented new-data interface. That changes the scoring workflow. Do not substitute novelty scores on the training rows for the fitted LOF scores above.

## Try One Small Modification

Fit a second LOF model with three neighbors. Keep the first model and input table unchanged, then compare the two score columns. Which observations have the largest scores under each neighborhood size?

In [ ]:
lof_three = LocalOutlierFactor(n_neighbors=3).fit(X)
comparison = pd.DataFrame(
    {"Two neighbors": lof_scores, "Three neighbors": -lof_three.negative_outlier_factor_},
    index=X.index,
)
display(records.join(comparison).round(3))
print("First maximum by setting (inspect ties in the table):")
print(comparison.idxmax())

With three neighbors, E, F, and G share the largest score, about 2.718. `idxmax()` reports the first maximum rather than listing all ties. Changing `n_neighbors` changes the reference neighborhood, so it requires a new fit. It does not merely move a cutoff on the old scores.

## Ready for 09b

Return to the lecture when you can explain the difference between constructing a model, fitting it, reading a fitted attribute, calling a scoring method, and selecting records. Keep the score sign and row alignment explicit each time.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).